# Tokenizers & Embeddings — Masterclass
**AI Engineering Core Track | Foundations before RAG**

**Prerequisites:** Python, NumPy basics, a Jupyter kernel. No deep learning experience required.

**Learning outcomes:** understand tokenization algorithms, inspect model-specific token IDs, explain embedding matrices and contextual representations, derive pooling and normalization, generate real embeddings using OpenAI, Hugging Face and Ollama, and understand their compatibility and operational constraints.

| Module | Subject |
|---|---|
| 0 | Environment and mental model |
| 1 | Tokenization fundamentals |
| 2 | BPE, WordPiece, Unigram and tokenizer training |
| 3 | Real tokenizers, masks, context limits |
| 4 | Embedding fundamentals and neural internals |
| 5 | OpenAI, Hugging Face and Ollama labs |
| 6 | Evaluation, engineering and troubleshooting |
| 7 | Interview revision and next-step teaser |

Run cells in order. Provider labs are optional and independently gated. Outputs are intentionally cleared; live calls may cost money. No RAG, vector database or semantic search implementation is included.

## 0 — Setup
Use your existing uv project in VS Code. In its terminal, from the folder containing `pyproject.toml`, run:
```powershell
uv add numpy pandas matplotlib python-dotenv tiktoken tokenizers transformers sentence-transformers torch openai requests huggingface-hub
uv add --dev ipykernel
```
Select the project's `.venv` Python as the notebook kernel. Alternatively uncomment the installation cell below in a standard notebook environment. The first model/tokenizer downloads require internet access and disk space. CPU is sufficient for the small local model used here.

### 0.1 Install (alternative to uv)

Run only if dependencies are missing. Restart the kernel if necessary. Avoid mixing pip and uv in the same project unnecessarily.

In [ ]:
# %pip install numpy pandas matplotlib python-dotenv tiktoken tokenizers transformers sentence-transformers torch openai requests huggingface-hub

### 0.2 Imports and configuration

Keep secrets in a `.env` file with `OPENAI_API_KEY=...` and `HF_TOKEN=...` (or `HUGGINGFACE_API_KEY=...`). Add `.env` to `.gitignore`. Never print keys. Public Hugging Face checkpoints usually do not require a key. Set the flags to True only for labs you intend to run. OpenAI calls are paid; Ollama requires a running local server.

In [ ]:
import os, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACE_API_KEY')
RUN_OPENAI = False
RUN_HF = True
RUN_OLLAMA = False
RUN_HF_HOSTED = False
print('OpenAI key available:', bool(os.getenv('OPENAI_API_KEY')))
print('HF key available:', bool(HF_TOKEN))

## 1 — The mental model
A computer represents text using numbers. But not every number represents meaning.

```text
Raw text → tokenizer → token IDs → embedding lookup → neural layers
                                                    ↓
                            contextual token vectors → pooling/projection → text vector
```
A generation model usually takes contextual states into a vocabulary prediction head instead. An embedding model typically returns a representation suitable for downstream numerical tasks. The tokenizer is not itself the neural embedding model.

| Concept | Example | Meaning |
|---|---|---|
| Token | `house` | Text fragment |
| Token ID | 1234 (illustrative) | Index in a vocabulary |
| Token vector | [0.2, -0.4, ...] | Learned numerical representation |
| Text vector | [0.08, 0.13, ...] | Representation of a whole input |

Token IDs are categorical indices, not measurements. ID 100 is not twice as meaningful as ID 50.

### 1.1 Characters, words and bytes

Python strings expose Unicode code points; UTF-8 stores bytes. Whitespace splitting is a crude word tokenizer. Neither is equivalent to a modern model tokenizer. Observe Tamil and emoji.

In [ ]:
texts = ['Hello world!', 'unbelievable', 'AI engineering 🚀', 'வணக்கம்', 'café', '  Hello\nworld']
display(pd.DataFrame([{'text':s,'code_points':len(s),'whitespace_words':len(s.split()),'utf8_bytes':len(s.encode('utf-8'))} for s in texts]))

### 1.2 Build a toy vocabulary

A vocabulary assigns IDs to tokens. Unknown words collapse to UNK in this toy word-level design. Modern subword or byte-based systems mitigate this. Real vocabularies are learned and fixed at inference.

In [ ]:
vocab = {'[UNK]':0,'a':1,'house':2,'is':3,'beautiful':4}
def toy_encode(s): return [vocab.get(w,0) for w in s.lower().split()]
for s in ['a house is beautiful','a villa is beautiful']:
    print(s, toy_encode(s))

## 2 — Tokenization algorithms
A tokenizer may include normalization, pre-tokenization, a learned segmentation model, post-processing and decoding. The exact pipeline is checkpoint-specific.

| Strategy | Core idea | Trade-off |
|---|---|---|
| Word | Whole words | Large vocabulary, unknown words |
| Character | Individual characters | Long sequences |
| Byte | Bytes from encoded text | Broad coverage, long sequences |
| BPE | Iteratively merge frequent adjacent units | Compact common patterns |
| WordPiece | Vocabulary learned using a likelihood-inspired criterion; commonly greedy longest-match inference | Continuation markers such as `##` |
| Unigram | Start with candidate pieces and prune using a probabilistic objective | Multiple possible segmentations |

SentencePiece is a tokenizer framework that supports algorithms including BPE and Unigram; it is not a fourth competing segmentation objective. Byte-level BPE starts from bytes rather than just characters. Normalization can alter case or Unicode forms; not every tokenizer preserves original text exactly.

### 2.1 Implement a tiny BPE trainer

Count adjacent pairs across a toy corpus, merge the most frequent pair, then repeat. This demonstration uses characters, not GPT-style byte-level preprocessing. The learned merge order is part of the tokenizer.

In [ ]:
from collections import Counter
corpus = Counter({'low':5,'lower':2,'newest':6,'widest':3})
words = {tuple(w):freq for w,freq in corpus.items()}
merges=[]
for step in range(8):
    pairs=Counter()
    for pieces,freq in words.items():
        for pair in zip(pieces,pieces[1:]): pairs[pair]+=freq
    if not pairs: break
    pair=max(pairs,key=pairs.get); merges.append(pair)
    updated=Counter()
    for pieces,freq in words.items():
        out=[]; i=0
        while i<len(pieces):
            if tuple(pieces[i:i+2])==pair: out.append(''.join(pair)); i+=2
            else: out.append(pieces[i]); i+=1
        updated[tuple(out)]+=freq
    words=dict(updated)
    print(step+1,pair,pairs[pair], list(words))

### 2.2 Apply learned merges to unseen text

Inference applies the learned merge order; it does not retrain the tokenizer on every sentence. This simplified encoder lacks unknown-character handling.

In [ ]:
def apply_merges(word):
    pieces=list(word)
    for pair in merges:
        out=[]; i=0
        while i<len(pieces):
            if tuple(pieces[i:i+2])==pair: out.append(''.join(pair)); i+=2
            else: out.append(pieces[i]); i+=1
        pieces=out
    return pieces
for w in ['lowest','newer','widest','unseen']: print(w,apply_merges(w))

### 2.3 Train three real subword tokenizers

Hugging Face Tokenizers provides implementations of BPE, WordPiece and Unigram. This tiny corpus is educational only. Different algorithms may produce different boundaries and IDs even with identical training text. The Unigram vocabulary size is a target, not a guarantee.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
training_texts=['learning machine learning','learn learned learner','token tokenizer tokenization','house houses housing','beautiful beautifully beauty']*10
specs=[('BPE',models.BPE(unk_token='[UNK]'),trainers.BpeTrainer(vocab_size=60,special_tokens=['[UNK]'])),('WordPiece',models.WordPiece(unk_token='[UNK]'),trainers.WordPieceTrainer(vocab_size=60,special_tokens=['[UNK]'])),('Unigram',models.Unigram(),trainers.UnigramTrainer(vocab_size=35,unk_token='[UNK]',special_tokens=['[UNK]']))]
for name,model,trainer in specs:
    tok=Tokenizer(model); tok.pre_tokenizer=pre_tokenizers.Whitespace()
    tok.train_from_iterator(training_texts,trainer=trainer)
    result=tok.encode('learning tokenization houses')
    print(name,result.tokens,result.ids)

## 3 — Production tokenizers
A model and tokenizer must share a compatible vocabulary, ID mapping, special tokens and preprocessing. Some models share the same tokenizer; model-dependent does not mean every model has a unique tokenizer. Never feed arbitrary IDs from another tokenizer into a model.

OpenAI's `tiktoken` is a local tokenizer library; it does not require an API key. Hugging Face `AutoTokenizer` loads a checkpoint's tokenizer configuration. Ollama generally handles tokenization internally; its embedding API does not expose the individual token IDs.

### 3.1 OpenAI: encode, decode and inspect bytes

Use `encoding_for_model` to choose the embedding model’s encoding. A token may contain leading spaces or only part of a UTF-8 character. Inspect raw bytes rather than assuming every token is independently decodable.

In [ ]:
import tiktoken
OPENAI_MODEL='text-embedding-3-small'
enc=tiktoken.encoding_for_model(OPENAI_MODEL)
s='Tokenization is fascinating! 🚀'
ids=enc.encode(s)
display(pd.DataFrame({'id':ids,'raw_bytes':[repr(enc.decode_single_token_bytes(i)) for i in ids]}))
print('Encoding:',enc.name,'count:',len(ids),'round trip:',enc.decode(ids)==s)

### 3.2 Compare encodings and languages

The same input can consume different numbers of tokens under different encodings. Compare cl100k_base and o200k_base, without assuming either is better for every language or model.

In [ ]:
other=tiktoken.get_encoding('o200k_base')
rows=[{'text':s,'characters':len(s),'cl100k_base':len(enc.encode(s)),'o200k_base':len(other.encode(s))} for s in texts]
display(pd.DataFrame(rows))
pd.DataFrame(rows).set_index('text')[['cl100k_base','o200k_base']].plot.bar(figsize=(9,3)); plt.ylabel('Tokens'); plt.tight_layout(); plt.show()

### 3.3 Hugging Face tokenizer

Use the tokenizer from the exact sentence embedding checkpoint used later. Inspect IDs, special tokens, offsets into the original string and the vocabulary size. The checkpoint uses a BERT-family WordPiece tokenizer.

In [ ]:
from transformers import AutoTokenizer
HF_MODEL='sentence-transformers/all-MiniLM-L6-v2'
if RUN_HF:
    hf_tok=AutoTokenizer.from_pretrained(HF_MODEL,token=HF_TOKEN,use_fast=True)
    sample='The house is beautifully renovated.'
    batch=hf_tok(sample,return_offsets_mapping=True,return_special_tokens_mask=True)
    display(pd.DataFrame({'token':hf_tok.convert_ids_to_tokens(batch['input_ids']),'id':batch['input_ids'],'offset':batch['offset_mapping'],'special':batch['special_tokens_mask']}))
    print('Vocabulary:',len(hf_tok),'Special tokens:',hf_tok.special_tokens_map)

### 3.4 Padding, truncation and attention masks

Batches need rectangular tensors. Padding fills unused positions, and attention masks usually mark real positions with 1 and padding with 0. Truncation removes content; it is not compression. Special tokens also consume positions.

In [ ]:
if RUN_HF:
    batch=hf_tok(['A house.','A large renovated house with a spacious garden.'],padding=True,truncation=True,max_length=10,return_tensors='pt')
    print('Input IDs:',batch['input_ids'])
    print('Attention mask:',batch['attention_mask'])
    print('Shape:',tuple(batch['input_ids'].shape))
    for ids_ in batch['input_ids']: print(hf_tok.convert_ids_to_tokens(ids_.tolist()))

### 3.5 Token budget and silent truncation

A model has an input limit; embedding and generation models may have different limits. Sentence Transformers can impose a shorter limit than its underlying encoder. Use a small artificial limit to observe lost content.

In [ ]:
if RUN_HF:
    long_text='A renovated property with new flooring. '*20+'The roof needs repair.'
    full=hf_tok(long_text,add_special_tokens=True,truncation=False)['input_ids']
    short=hf_tok(long_text,truncation=True,max_length=32)['input_ids']
    print('Original:',len(full),'Retained:',len(short))
    print(hf_tok.decode(short,skip_special_tokens=True))

## 4 — Embeddings: from IDs to meaning
An embedding is a dense numerical representation. Sparse one-hot vectors identify categories but do not directly encode learned relationships. A neural embedding table has shape `(vocabulary_size, hidden_size)`; looking up N IDs produces `(N, hidden_size)`.

Three different concepts are often called embeddings:
- **Input token embeddings:** learned rows in a lookup matrix, before contextual processing.
- **Contextual token representations:** output vectors influenced by surrounding tokens and positions.
- **Text embeddings:** one vector for a sentence, paragraph or document, typically derived from contextual states through pooling and sometimes projection or normalization.

A tokenizer produces IDs, not semantic vectors. There is no separate required process called tokenizer embedding when using a text embedding API.

### 4.1 One-hot versus embedding lookup

This toy embedding matrix contains invented random values, not learned semantics. Multiplying a one-hot vector by the embedding matrix is equivalent to selecting a row. Vocabulary size and embedding dimension are independent.

In [ ]:
rng=np.random.default_rng(42)
E=rng.normal(size=(6,4))
ids=np.array([1,3,1])
one_hot=np.eye(6)[ids]
print('One-hot shape:',one_hot.shape,'Embedding shape:',E[ids].shape)
print(E[ids]); assert np.allclose(one_hot@E,E[ids])
plt.imshow(E,cmap='coolwarm',aspect='auto'); plt.colorbar(); plt.xlabel('Embedding dimension'); plt.ylabel('Token ID'); plt.show()

### 4.2 How learned embeddings change

A minimal gradient-descent illustration: adjust a single row toward an artificial target. Real models learn through language modeling, contrastive objectives or other training losses across massive datasets; this is not a realistic embedding training recipe.

In [ ]:
W=E.copy(); target=np.array([0.5,-0.2,0.3,0.8]); losses=[]
for _ in range(30):
    error=W[1]-target; losses.append(np.mean(error**2)); W[1]-=0.2*(2*error/len(error))
plt.plot(losses); plt.xlabel('Training step'); plt.ylabel('Toy MSE'); plt.show()
print('Updated row:',W[1])

### 4.3 Static vs contextual embeddings
Word2Vec and GloVe classically learn static vectors per word. Transformer models use attention and positional information to contextualize tokens. The word “bank” in “river bank” and “bank loan” starts with the same lookup vector if tokenized identically, but the contextual vector can differ.

Self-attention forms query, key and value representations. In simplified notation: `softmax(QKᵀ / sqrt(d_k)) V`. Position information can be learned absolute embeddings, rotary position transformations or another architecture-specific mechanism. Encoder models usually allow bidirectional attention; causal generation models generally restrict attention to earlier positions. Neither attention weights nor embedding coordinates are guaranteed to be human-interpretable.

### 4.4 Inspect real neural token representations

Load the underlying encoder of MiniLM. Compare the initial token lookup and last hidden state for the same token in two contexts. `eval()` and inference mode disable training-time behavior and gradients. Hidden-state distances here illustrate contextualization, not a search system.

In [ ]:
import torch
from transformers import AutoModel
if RUN_HF:
    encoder=AutoModel.from_pretrained(HF_MODEL,token=HF_TOKEN).eval()
    sentences=['The bank approved the loan.','The bank of the river was muddy.']
    inputs=hf_tok(sentences,padding=True,return_tensors='pt')
    bank_id=hf_tok.convert_tokens_to_ids('bank')
    positions=[(row==bank_id).nonzero(as_tuple=True)[0][0].item() for row in inputs['input_ids']]
    with torch.inference_mode():
        initial=encoder.get_input_embeddings()(inputs['input_ids'])
        contextual=encoder(**inputs).last_hidden_state
    print('Lookup:',tuple(initial.shape),'Contextual:',tuple(contextual.shape))
    print('Initial difference:',torch.linalg.vector_norm(initial[0,positions[0]]-initial[1,positions[1]]).item())
    print('Contextual difference:',torch.linalg.vector_norm(contextual[0,positions[0]]-contextual[1,positions[1]]).item())

### 4.5 Pooling: many tokens to one vector

Mean pooling averages valid token states, excluding padding using the mask. Other methods include CLS pooling, max pooling, last-token pooling and learned pooling. The correct choice depends on how the model was trained. MiniLM uses mean pooling. Do not substitute pooling arbitrarily.

In [ ]:
def masked_mean(hidden, mask):
    mask=mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
if RUN_HF:
    pooled=masked_mean(contextual,inputs['attention_mask'])
    print('Before pooling:',tuple(contextual.shape),'After:',tuple(pooled.shape))
    print(pooled[0,:8])

### 4.6 Vector magnitude and normalization

L2 normalization divides each vector by its Euclidean norm. Unit-length vectors have norm 1; zero vectors require special handling. Normalization changes magnitude, not direction. Not every embedding model requires normalization.

In [ ]:
v=np.array([3.,4.]); norm=np.linalg.norm(v); unit=v/norm
print('Original:',v,'norm:',norm,'unit:',unit)
if RUN_HF:
    normalized=torch.nn.functional.normalize(pooled,p=2,dim=1)
    print('Before:',pooled.norm(dim=1),'After:',normalized.norm(dim=1))

### 4.7 How text embedding models are trained
A common contrastive training approach learns representations for related input pairs while distinguishing unrelated pairs. Supervision may come from human annotations, mined pairs, synthetic pairs or other objectives. A generation model is typically optimized for next-token prediction; extracting its hidden states does not automatically make it a high-quality text embedding model.

Dimensions are the number of coordinates, not token count or number of concepts. A 384-dimensional vector is not inherently worse than a 1536-dimensional vector. Training quality, language coverage, domain fit and input handling matter. Coordinates are not usually named features such as `bedrooms` or `condition`.

## 5 — Generate actual text embeddings across providers
Use a common set of inputs for shape, norm and latency inspection only. Do not compare coordinate values between different model spaces. OpenAI is hosted; Hugging Face can run locally or via a supported hosted provider; Ollama runs through a local service in this lab.

### 5.1 Shared inputs and inspection utility

Use public, non-sensitive sample sentences. The inspection function checks shape, finiteness and magnitude without calculating pairwise similarity.

In [ ]:
samples=['A renovated house with a new kitchen.','The property needs roof repairs.','Machine learning uses numerical representations.','வணக்கம்']
results={}; timings={}
def inspect_vectors(name, vectors):
    arr=np.asarray(vectors,dtype=np.float32)
    assert arr.ndim==2 and arr.shape[0]==len(samples)
    assert np.isfinite(arr).all()
    results[name]=arr
    display(pd.DataFrame({'text':samples,'dimensions':arr.shape[1],'L2_norm':np.linalg.norm(arr,axis=1),'first_5_coordinates':[row[:5].round(5).tolist() for row in arr]}))

### 5.2 OpenAI embeddings API

Set RUN_OPENAI=True in setup to execute. The SDK sends strings; OpenAI tokenizes internally. `text-embedding-3-small` defaults to 1536 dimensions; `text-embedding-3-large` defaults to 3072. The v3 API supports requesting fewer dimensions. The response includes token usage. The key is not needed for tiktoken but is needed here.

In [ ]:
from openai import OpenAI
if RUN_OPENAI:
    assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'
    client=OpenAI()
    start=time.perf_counter()
    response=client.embeddings.create(model=OPENAI_MODEL,input=samples)
    timings['OpenAI']=time.perf_counter()-start
    inspect_vectors('OpenAI',[item.embedding for item in sorted(response.data,key=lambda x:x.index)])
    print('Usage:',response.usage)
else: print('OpenAI lab disabled')

### 5.3 OpenAI configurable output dimension

Request 256 dimensions explicitly. Treat this as a distinct embedding configuration. Do not mix vectors with different dimensions; smaller output dimensions can reduce storage but must be validated for quality. This is an additional paid request.

In [ ]:
RUN_DIMENSION_EXPERIMENT=False
if RUN_OPENAI and RUN_DIMENSION_EXPERIMENT:
    reduced=client.embeddings.create(model=OPENAI_MODEL,input=samples,dimensions=256)
    inspect_vectors('OpenAI-256',[item.embedding for item in sorted(reduced.data,key=lambda x:x.index)])

### 5.4 Hugging Face local Sentence Transformers

This wrapper applies the checkpoint-specific tokenizer, encoder and pooling modules. The selected MiniLM checkpoint returns 384-dimensional vectors and has a default Sentence Transformers maximum sequence length of 256 tokens. It is primarily English-oriented; the Tamil sample is illustrative, not evidence of multilingual quality.

In [ ]:
from sentence_transformers import SentenceTransformer
if RUN_HF:
    st=SentenceTransformer(HF_MODEL,token=HF_TOKEN,device='cpu')
    print('Max sequence length:',st.max_seq_length,'Embedding dimension:',st.get_sentence_embedding_dimension())
    start=time.perf_counter()
    hf_vectors=st.encode(samples,batch_size=4,normalize_embeddings=True,convert_to_numpy=True)
    timings['HF local']=time.perf_counter()-start
    inspect_vectors('HF local',hf_vectors)

### 5.5 Reconstruct the Hugging Face pipeline

Compare manual encoder → masked mean → L2 normalization to the Sentence Transformers wrapper for this particular model. The results should be close, subject to floating-point differences. For other checkpoints, additional modules or prompts may be required.

In [ ]:
if RUN_HF:
    inp=hf_tok(samples,padding=True,truncation=True,max_length=st.max_seq_length,return_tensors='pt')
    with torch.inference_mode():
        h=encoder(**inp).last_hidden_state
        manual=torch.nn.functional.normalize(masked_mean(h,inp['attention_mask']),dim=1).numpy()
    print('Manual shape:',manual.shape)
    print('Maximum absolute coordinate difference:',np.max(np.abs(manual-hf_vectors)))

### 5.6 Optional Hugging Face hosted inference

A Hugging Face key authorizes access but does not guarantee a hosted provider supports a checkpoint. Set HF_HOSTED_MODEL in .env to a feature-extraction model currently supported by your provider, and enable RUN_HF_HOSTED. Hosted feature extraction may return token-level or text-level arrays depending on endpoint; inspect before assuming pooling. Calls may be billed.

In [ ]:
from huggingface_hub import InferenceClient
if RUN_HF_HOSTED:
    assert HF_TOKEN and os.getenv('HF_HOSTED_MODEL'), 'Configure HF_TOKEN and HF_HOSTED_MODEL'
    hf_client=InferenceClient(token=HF_TOKEN)
    output=hf_client.feature_extraction(samples[0],model=os.environ['HF_HOSTED_MODEL'])
    print('Hosted output shape:',np.asarray(output).shape)
else: print('Hosted HF lab disabled')

### 5.7 Ollama local embeddings

Install Ollama separately and run `ollama pull embeddinggemma` in your terminal. Ensure Ollama is running (normally port 11434). Enable RUN_OLLAMA in setup. The `/api/embed` endpoint accepts a batch of strings; tokenization and pooling happen inside the runtime. `truncate=False` requests an error instead of silent truncation. No cloud key is needed for this local endpoint.

In [ ]:
import requests
OLLAMA_MODEL='embeddinggemma'
if RUN_OLLAMA:
    start=time.perf_counter()
    response=requests.post('http://localhost:11434/api/embed',json={'model':OLLAMA_MODEL,'input':samples,'truncate':False},timeout=180)
    response.raise_for_status(); payload=response.json()
    timings['Ollama']=time.perf_counter()-start
    inspect_vectors('Ollama',payload['embeddings'])
    print('Reported prompt tokens:',payload.get('prompt_eval_count'))
else: print('Ollama lab disabled')

### 5.8 Provider comparison

Compare measured shapes and latency only. This is a single small batch, not a controlled benchmark: hardware, network, cold starts, model loading and provider implementation differ. Model download time is not included in the HF timer.

In [ ]:
summary=pd.DataFrame([{'provider':name,'vectors':arr.shape[0],'dimension':arr.shape[1],'dtype':str(arr.dtype),'raw_bytes_per_vector':arr.shape[1]*arr.dtype.itemsize,'observed_seconds':timings.get(name,np.nan)} for name,arr in results.items()])
display(summary)

## 6 — Engineering considerations
**Compatibility contract:** record the model identifier and version, tokenizer/preprocessing revision, pooling, normalization, dimension and any task-specific prefixes. Some models are trained to accept different prefixes for different tasks; follow their model card. Equal dimensions alone do not imply compatible vectors. Upgrading a checkpoint or changing configuration may require regenerating all comparable embeddings. The generation LLM can be a different model because it has its own tokenizer and embedding layers.

**Input length:** long text may be truncated or rejected. Inspect exact tokenizer counts when available. A document embedding is simply an embedding of document text (or a derived representation of its pieces), not a separate tokenizer algorithm. Chunking is typically applied before final embedding; token-aware chunkers can use a tokenizer while choosing boundaries. This notebook deliberately leaves chunking pipelines for another lesson.

### 6.1 Raw vector storage estimates

Float32 uses four bytes per coordinate. Estimate uncompressed payload only; actual storage includes metadata, serialization, indexing and system overhead. No database is used.

In [ ]:
rows=[]
for d in [256,384,768,1536,3072]:
    rows.append({'dimension':d,'bytes_per_vector':4*d,'GiB_for_1_million':4*d*1_000_000/2**30})
display(pd.DataFrame(rows))

### 6.2 Token-based cost estimator

Enter your actual price per million input tokens from the provider pricing page. No live price is assumed. This estimate excludes repeated requests and other charges.

In [ ]:
token_count=sum(len(enc.encode(s)) for s in samples)
PRICE_PER_MILLION=None  # replace with your actual USD rate
print('Estimated input tokens:',token_count)
if PRICE_PER_MILLION is not None: print('Estimated USD:',token_count*PRICE_PER_MILLION/1_000_000)

### 6.3 Validate embedding output

Check the contract at the boundary of an application: correct row count, expected dimension, numeric finite values and optional unit norms. This is structural validation, not semantic quality evaluation.

In [ ]:
def validate_embeddings(vectors, expected_rows, expected_dim, unit_norm=False):
    a=np.asarray(vectors)
    if a.shape!=(expected_rows,expected_dim): raise ValueError(f'Unexpected shape: {a.shape}')
    if not np.issubdtype(a.dtype,np.number) or not np.isfinite(a).all(): raise ValueError('Invalid numeric values')
    if unit_norm and not np.allclose(np.linalg.norm(a,axis=1),1,atol=1e-4): raise ValueError('Unexpected norms')
    return True
for name,a in results.items():
    expected={'OpenAI':1536,'OpenAI-256':256,'HF local':384}.get(name,a.shape[1])
    print(name,validate_embeddings(a,len(samples),expected,unit_norm=name=='HF local'))

### 6.4 Troubleshooting and model selection
| Symptom | What to check |
|---|---|
| 401/403 | Key, permissions, provider access |
| 429 | Rate limits, quota, retry policy |
| Ollama connection refused | Server running and correct endpoint |
| Model not found | Exact identifier, download or hosted availability |
| Different vector dimensions | Model/configuration mismatch |
| Missing meaning from long input | Truncation, language coverage, domain fit |
| Out of memory | Smaller batch/model, CPU/GPU memory |
| Same input slightly different | Runtime precision, version, inference mode |

For production, record model provenance, batch inputs, use bounded retries on transient errors, monitor failure rates, and validate representative data. For quality evaluation, use task-specific labeled examples, language/domain coverage, and downstream metrics. Norms and shape alone cannot establish usefulness.

## 7 — Interview revision and practice
Try answering before reading the answers.

1. **Can a token ID encode meaning?** No; it is a vocabulary index. Learned neural representations encode useful patterns.
2. **Can two models share a tokenizer?** Yes, if the full tokenizer contract is compatible.
3. **Are token and text embeddings identical?** No; text embeddings often aggregate contextual token states.
4. **Why does the same word have different contextual vectors?** Neural layers incorporate surrounding context and position.
5. **Why use masks in mean pooling?** To prevent padding from distorting the vector.
6. **Is a larger dimension always better?** No; quality must be validated against the intended task.
7. **Can two 384-dimensional models be used interchangeably?** Not merely because their dimensions match.
8. **Does an OpenAI embedding request require manual tokenization?** Not for string input; the service handles it.
9. **Does the generation LLM need the embedding model's tokenizer?** No; each model has its own compatible tokenizer.
10. **What happens to text beyond the input limit?** Depending on configuration, it is rejected or truncated.

**Practice:** compare 10 multilingual sentences across tokenizers; change the padding and max length; inspect the same word in three contexts; reconstruct pooling manually; calculate vector storage for your own data volume. Explain each result in your own words.

## 8 — What comes after embeddings? (teaser only)
Once a model converts text into a fixed-length numerical vector, the vector can be persisted with its model metadata and used in downstream machine learning, clustering, classification, or similarity-based retrieval. A future lesson will explain distance metrics, vector indexes, semantic search and RAG — no implementation here.

```text
This masterclass                 Future modules
Text → tokens → IDs → vectors  |  storage → similarity/indexing → retrieval → applications
```
The key takeaway: **tokenization determines how text enters the model; the embedding model determines the numerical representation; the downstream task determines how that representation is used.**

### Official reading
- [OpenAI tokenizer](https://platform.openai.com/tokenizer) · [Embeddings guide](https://platform.openai.com/docs/guides/embeddings) · [tiktoken](https://github.com/openai/tiktoken)
- [Hugging Face tokenizers](https://huggingface.co/docs/tokenizers/index) · [Transformers tokenizer API](https://huggingface.co/docs/transformers/main_classes/tokenizer) · [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [Sentence Transformers](https://www.sbert.net/) · [HF InferenceClient](https://huggingface.co/docs/huggingface_hub/package_reference/inference_client)
- [Ollama embedding API](https://docs.ollama.com/api/embed) · [EmbeddingGemma](https://ollama.com/library/embeddinggemma)